[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/crewai/crewai_ parallelization_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>
<h1 align="center">使用 Arize Phoenix 追踪 CrewAI - 并行工作流</h1>

In [ ]:
!pip install -q arize-phoenix opentelemetry-sdk opentelemetry-exporter-otlp crewai crewai_tools openinference-instrumentation-crewai

# 设置密钥与依赖

注意：运行此 Colab 需要：

*   OpenAI API 密钥 (https://openai.com/)
*   Serper API 密钥 (https://serper.dev/)
*   Phoenix API 密钥 (https://app.phoenix.arize.com/)

In [ ]:
import os
from getpass import getpass

# 尚未设置 API 密钥时提示用户输入
openai_key = os.getenv("OPENAI_API_KEY", "OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY", "SERPER_API_KEY")

if openai_key == "OPENAI_API_KEY":
    openai_key = getpass("请输入 OPENAI_API_KEY： ")

if serper_key == "SERPER_API_KEY":
    serper_key = getpass("请输入 SERPER_API_KEY： ")

# 使用提供的密钥设置环境变量
os.environ["OPENAI_API_KEY"] = openai_key
os.environ["SERPER_API_KEY"] = serper_key

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="crewai-agents")

# 为 CrewAI 接入插桩

In [ ]:
from openinference.instrumentation.crewai import CrewAIInstrumentor

CrewAIInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

## 定义智能体

In [ ]:
from crewai import Agent, Crew, Task
from crewai.process import Process

researcher_1 = Agent(
    role="LLM 研究员 A",
    goal="研究 AI 趋势 1，并清楚总结。",
    backstory="专长是模型安全与治理。",
    verbose=True,
)

researcher_2 = Agent(
    role="LLM 研究员 B",
    goal="研究 AI 趋势 2，并清楚总结。",
    backstory="多模态与前沿模型专家。",
    verbose=True,
)

researcher_3 = Agent(
    role="LLM 研究员 C",
    goal="研究 AI 趋势 3，并清楚总结。",
    backstory="专注 AI 政策与对齐。",
    verbose=True,
)

aggregator = Agent(
    role="Aggregator",
    goal="将所有研究合并为一份综合总结报告。",
    backstory="擅长总结多个来源的信息架构师。",
    verbose=True,
)

## 定义任务

In [ ]:
# 定义并行研究任务
task1 = Task(
    description="总结 AI 安全与模型对齐的一项重要趋势。",
    expected_output="趋势 1 的简洁摘要",
    agent=researcher_1,
)

task2 = Task(
    description="总结多模态或前沿 AI 系统的一项关键创新。",
    expected_output="趋势 2 的简洁摘要",
    agent=researcher_2,
)

task3 = Task(
    description="总结 AI 政策、监管或社会影响的当前议题。",
    expected_output="趋势 3 的简洁摘要",
    agent=researcher_3,
)

# 汇总任务
aggregation_task = Task(
    description="将三份 AI 趋势摘要合并为一份连贯报告。",
    expected_output="涵盖三种趋势的综合报告。",
    agent=aggregator,
)

## 创建 Crew

In [ ]:
crew = Crew(
    agents=[researcher_1, researcher_2, researcher_3, aggregator],
    tasks=[task1, task2, task3, aggregation_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print(result)

### 在 Phoenix 项目中查看运行产生的追踪与跨度。